# Rotate And Crop Video Batch

Batch-process all concatenated behavior camera videos in a folder using the same rotation and crop settings.

In [ ]:
from pathlib import Path
import math
import ffmpeg
import pandas as pd
from IPython.display import Image, Markdown, display
from tqdm import tqdm


def probe_video(video_path):
    probe = ffmpeg.probe(str(video_path))
    video_stream = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    width = int(video_stream['width'])
    height = int(video_stream['height'])
    duration = float(video_stream['duration'])
    codec_name = video_stream.get('codec_name', 'Unknown')
    return {
        'width': width,
        'height': height,
        'duration': duration,
        'codec_name': codec_name,
    }


def compute_rotated_canvas(width, height, angle_degrees):
    angle_rad = math.radians(angle_degrees)
    new_width = int(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
    new_height = int(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))
    return angle_rad, new_width, new_height


def compute_crop_box(vertices, rotated_width, rotated_height):
    x_coordinates = [v[0] for v in vertices]
    y_coordinates = [v[1] for v in vertices]

    x_min = max(0, min(x_coordinates))
    x_max = min(rotated_width, max(x_coordinates))
    y_min = max(0, min(y_coordinates))
    y_max = min(rotated_height, max(y_coordinates))

    crop_x = x_min
    crop_y = y_min
    crop_width = x_max - x_min
    crop_height = y_max - y_min

    if crop_width <= 0 or crop_height <= 0:
        raise ValueError(
            f'Invalid crop dimensions: {crop_width}x{crop_height} at ({crop_x}, {crop_y}).'
        )

    return crop_x, crop_y, crop_width, crop_height


def build_output_path(input_path, output_dir, suffix='_rotated_cropped'):
    return output_dir / f'{input_path.stem}{suffix}.avi'


def build_filtered_stream(input_path, angle_degrees, vertices, preview_time_sec=None):
    info = probe_video(input_path)
    angle_rad, rotated_width, rotated_height = compute_rotated_canvas(
        info['width'], info['height'], angle_degrees
    )
    crop_x, crop_y, crop_width, crop_height = compute_crop_box(
        vertices, rotated_width, rotated_height
    )

    input_kwargs = {}
    if preview_time_sec is not None:
        input_kwargs['ss'] = max(0, float(preview_time_sec))

    stream = (
        ffmpeg
        .input(str(input_path), **input_kwargs)
        .filter(
            'pad',
            rotated_width,
            rotated_height,
            (rotated_width - info['width']) // 2,
            (rotated_height - info['height']) // 2,
            color='0xFFFFFF'
        )
        .filter('rotate', str(angle_rad))
        .filter('crop', crop_width, crop_height, crop_x, crop_y)
    )

    transform_info = {
        'duration_sec': info['duration'],
        'input_codec': info['codec_name'],
        'rotated_width': rotated_width,
        'rotated_height': rotated_height,
        'crop_x': crop_x,
        'crop_y': crop_y,
        'crop_width': crop_width,
        'crop_height': crop_height,
    }
    return stream, transform_info


def render_preview_frame(input_path, angle_degrees, vertices, preview_time_fraction=0.5):
    _, info = build_filtered_stream(input_path, angle_degrees, vertices)
    preview_time_sec = info['duration_sec'] * preview_time_fraction
    stream, info = build_filtered_stream(
        input_path,
        angle_degrees,
        vertices,
        preview_time_sec=preview_time_sec,
    )

    out, err = (
        stream
        .output('pipe:', vframes=1, format='image2', vcodec='png')
        .global_args('-loglevel', 'error')
        .run(capture_stdout=True, capture_stderr=True)
    )

    if not out:
        error_text = err.decode('utf-8', errors='replace').strip()
        raise RuntimeError(error_text or f'Could not render preview frame for {input_path.name}')

    info['preview_time_sec'] = preview_time_sec
    info['image_bytes'] = out
    return info


def preview_batch_transforms(input_dir, angle_degrees, vertices, pattern='*.mp4', preview_time_fraction=0.5):
    input_dir = Path(input_dir)
    video_files = sorted(input_dir.glob(pattern))
    if not video_files:
        raise FileNotFoundError(f'No files matching {pattern!r} found in {input_dir}')

    results = []
    for input_path in tqdm(video_files, desc='Preview frames', unit='file'):
        preview = render_preview_frame(
            input_path,
            angle_degrees=angle_degrees,
            vertices=vertices,
            preview_time_fraction=preview_time_fraction,
        )
        display(Markdown(
            f"### {input_path.name}\n"
            f"Preview time: `{preview['preview_time_sec']:.2f}s`  \n"
            f"Rotated canvas: `{preview['rotated_width']} x {preview['rotated_height']}`  \n"
            f"Crop box: `x={preview['crop_x']}, y={preview['crop_y']}, w={preview['crop_width']}, h={preview['crop_height']}`"
        ))
        display(Image(data=preview['image_bytes']))
        results.append({
            'input_file': str(input_path),
            'preview_time_sec': preview['preview_time_sec'],
            'rotated_width': preview['rotated_width'],
            'rotated_height': preview['rotated_height'],
            'crop_x': preview['crop_x'],
            'crop_y': preview['crop_y'],
            'crop_width': preview['crop_width'],
            'crop_height': preview['crop_height'],
        })

    return pd.DataFrame(results)


def rotate_and_crop_video(input_path, output_path, angle_degrees, vertices, overwrite=False):
    stream, info = build_filtered_stream(input_path, angle_degrees, vertices)

    ffmpeg_command = (
        stream
        .output(str(output_path), vcodec='mjpeg', qscale=3)
        .global_args('-progress', 'pipe:1', '-nostats', '-loglevel', 'error')
    )

    process = ffmpeg_command.run_async(
        pipe_stdout=True,
        pipe_stderr=True,
        overwrite_output=overwrite,
    )

    pbar = tqdm(
        total=info['duration_sec'],
        desc=input_path.name,
        unit='s',
        dynamic_ncols=True,
        leave=False,
    )

    for raw_line in process.stdout:
        line = raw_line.decode('utf-8', errors='replace').strip()
        if line.startswith('out_time_ms='):
            out_time_ms = int(line.split('=', 1)[1])
            current_time = out_time_ms / 1_000_000
            pbar.update(max(0, current_time - pbar.n))
        elif line == 'progress=end':
            break

    pbar.close()
    return_code = process.wait()
    stderr_output = process.stderr.read().decode('utf-8', errors='replace').strip()

    if return_code != 0:
        raise RuntimeError(stderr_output or f'ffmpeg failed with exit code {return_code}')

    return {
        'input_file': str(input_path),
        'output_file': str(output_path),
        'duration_sec': info['duration_sec'],
        'input_codec': info['codec_name'],
        'rotated_width': rotated_width,
        'rotated_height': rotated_height,
        'crop_x': crop_x,
        'crop_y': crop_y,
        'crop_width': crop_width,
        'crop_height': crop_height,
    }


def batch_rotate_and_crop(input_dir, output_dir, angle_degrees, vertices, pattern='*.mp4', overwrite=False):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    video_files = sorted(input_dir.glob(pattern))
    if not video_files:
        raise FileNotFoundError(f'No files matching {pattern!r} found in {input_dir}')

    results = []
    for input_path in tqdm(video_files, desc='Files', unit='file'):
        output_path = build_output_path(input_path, output_dir)

        if output_path.exists() and not overwrite:
            results.append({
                'input_file': str(input_path),
                'output_file': str(output_path),
                'status': 'skipped_existing',
            })
            continue

        try:
            result = rotate_and_crop_video(
                input_path=input_path,
                output_path=output_path,
                angle_degrees=angle_degrees,
                vertices=vertices,
                overwrite=overwrite,
            )
            result['status'] = 'processed'
            results.append(result)
        except Exception as exc:
            results.append({
                'input_file': str(input_path),
                'output_file': str(output_path),
                'status': 'error',
                'error': str(exc),
            })

    return pd.DataFrame(results)


In [ ]:
input_dir = Path('/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/YZ/Miniscope_data/Miniscope_data/Linear_track/BehavCamConcactenated_992')
output_dir = input_dir / 'rotated_and_cropped_avi'

# Use the same parameters you tested in the single-video notebook.
angle_to_rotate = 30
crop_vertices = [(51, 373), (709, 373), (709, 404), (51, 404)]
overwrite_existing = False
preview_time_fraction = 0.5

video_files = sorted(input_dir.glob('*.mp4'))
print(f'Found {len(video_files)} input videos in {input_dir}')
print(f'Output directory: {output_dir}')
video_files[:5]

In [ ]:
preview_results = preview_batch_transforms(
    input_dir=input_dir,
    angle_degrees=angle_to_rotate,
    vertices=crop_vertices,
    pattern='*.mp4',
    preview_time_fraction=preview_time_fraction,
)

preview_results

In [ ]:
batch_results = batch_rotate_and_crop(
    input_dir=input_dir,
    output_dir=output_dir,
    angle_degrees=angle_to_rotate,
    vertices=crop_vertices,
    pattern='*.mp4',
    overwrite=overwrite_existing,
)

batch_results